In [4]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
print("Imported all libraries!")

Imported all libraries!


## Loading the data

In [5]:
df = None

if os.path.exists("./data/df.parquet"): # checks if parquet file exists
    print("./data/df.parquet exists")
    df = pd.read_parquet("./data/df.parquet")
else: # create the parquet file bc man ts takes too long
    print("./data/df.parquet doesn't exist")
    dataframes = []

    for i in range(4): #loading data
        dataframes.append(pd.read_json(f"./data/dblp-ref-{i}.json", lines=True))
        print(f"dblp-ref-{i}.json loaded successfully!")

    df = pd.concat(dataframes, ignore_index=True)
    df.to_parquet("./data/df.parquet")

print("dataframe loaded successfully")
df.head()

./data/df.parquet exists
dataframe loaded successfully


,abstract,authors,n_citation,references,title,venue,year,id
0,The purpose of this study is to develop a lear...,"[Makoto Satoh, Ryo Muramatsu, Mizue Kayama, Ka...",0,"[51c7e02e-f5ed-431a-8cf5-f761f266d4be, 69b625b...",Preliminary Design of a Network Protocol Learn...,international conference on human-computer int...,2013,00127ee2-cb05-48ce-bc49-9de556b93346
1,This paper describes the design and implementa...,"[Gareth Beale, Graeme Earl]",50,"[10482dd3-4642-4193-842f-85f3b70fcf65, 3133714...",A methodology for the physically accurate visu...,visual analytics science and technology,2011,001c58d3-26ad-46b3-ab3a-c1e557d16821
2,This article applied GARCH model instead AR or...,"[Altaf Hossain, Faisal Zaman, Mohammed Nasser,...",50,"[2d84c0f2-e656-4ce7-b018-90eda1c132fe, a083a1b...","Comparison of GARCH, Neural Network and Suppor...",pattern recognition and machine intelligence,2009,001c8744-73c4-4b04-9364-22d31a10dbf1
3,NaN,"[Jea-Bum Park, Byungmok Kim, Jian Shen, Sun-Yo...",0,"[8c78e4b0-632b-4293-b491-85b1976675e6, 9cdc54f...",Development of Remote Monitoring and Control D...,,2011,00338203-9eb3-40c5-9f31-cbac73a519ec
4,NaN,"[Giovanna Guerrini, Isabella Merlo]",2,None,Reasonig about Set-Oriented Methods in Object ...,,1998,0040b022-1472-4f70-a753-74832df65266


## A1: Preprocessing

In [6]:
# normalize venue/title strings
df['venue'] = df['venue'].str.strip().str.lower()
df['title'] = df['title'].str.strip().str.lower()

# diagnostic: check ACL candidates BEFORE dropping rows with no abstract
# (ACL papers may exist in the data but lack abstracts)
acl_candidates = df[df['venue'].str.contains(
    'computational linguistics|association for computational|natural language proc', na=False, regex=True
)]['venue'].value_counts()
print("ACL-like venues (before abstract filter):\n", acl_candidates.head(10), "\n")

# drop nulls & empty strings
df = df.dropna(subset=['venue', 'title', 'abstract'])
df = df[(df['venue'] != "") & (df['title'] != "")]
print(f"After null/empty drop: {len(df)} rows")

# parquet round-trips list columns as numpy arrays — accept any sequence
df = df[df['authors'].apply(lambda x: x is not None and hasattr(x, '__len__') and len(x) > 0)]
df = df[df['venue'] != 'arxiv']
df = df.drop_duplicates(subset=['id'])
print(f"After authors/arXiv/dedup: {len(df)} rows")

# confirmed venue strings from DBLP V10 (full English names, lowercase)
# if ACL-like venues printed 0 above, we swap in EMNLP or drop it
acl_venue = acl_candidates.index[0] if len(acl_candidates) > 0 else None

target_venues = [
    'international conference on machine learning',   # ICML
    'knowledge discovery and data mining',            # KDD
    'international conference on management of data', # SIGMOD
    'computer vision and pattern recognition',        # CVPR (confirmed)
    'very large data bases',                          # VLDB
]
if acl_venue:
    target_venues.append(acl_venue)
    print(f"Using ACL venue: '{acl_venue}'")
else:
    # ACL has no abstracts in this dataset — substitute EMNLP
    nlp_alt = df[df['venue'].str.contains('empirical methods|emnlp', na=False, regex=True)]['venue'].value_counts()
    print("No ACL found. NLP alternatives:\n", nlp_alt.head(5))
    if len(nlp_alt) > 0:
        target_venues.append(nlp_alt.index[0])
        print(f"Using NLP substitute: '{nlp_alt.index[0]}'")

print(f"\nTarget venues: {target_venues}\n")

df_filtered = df[df['venue'].isin(target_venues)]
print(f"Papers found per venue:\n{df_filtered['venue'].value_counts()}\n")

if len(df_filtered) == 0:
    print("STILL EMPTY — top 30 actual venue strings:\n", df['venue'].value_counts().head(30))
    raise RuntimeError("No papers matched — update target_venues using the list above.")

df = df_filtered

# sample up to 3000 papers per venue (~18k total)
sampled = []
for venue in target_venues:
    subset = df[df['venue'] == venue]
    if len(subset) > 0:
        sampled.append(subset.sample(n=min(len(subset), 3000), random_state=42))
df = pd.concat(sampled, ignore_index=True)
print(f"After sampling — papers per venue:\n{df['venue'].value_counts()}")
print(f"Total: {len(df)}\n")

# apply TF-IDF
df['text'] = df['title'] + ' ' + df['abstract']
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', min_df=5)
tfidf_matrix = vectorizer.fit_transform(df['text'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

sp.save_npz('./data/tfidf_matrix.npz', tfidf_matrix)
import pickle
with open('./data/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# paper level
df = df.reset_index(drop=True)
df.to_parquet('./data/df_clean.parquet')

# author-level
author_df = df.explode('authors').rename(columns={'authors': 'author'})
author_df = author_df.groupby('author').agg(
    paper_count   = ('id', 'count'),
    avg_citations = ('n_citation', 'mean'),
    venues        = ('venue', lambda x: list(x.unique())),
    year_min      = ('year', 'min'),
    year_max      = ('year', 'max')
).reset_index()
author_df['venue_count'] = author_df['venues'].apply(len)
print(f"Author-level shape: {author_df.shape}")

# venue-level
venue_df = df.groupby('venue').agg(
    paper_count   = ('id', 'count'),
    avg_citations = ('n_citation', 'mean'),
    year_min      = ('year', 'min'),
    year_max      = ('year', 'max'),
    top_authors   = ('authors', lambda x: pd.Series([a for sublist in x for a in sublist]).value_counts().index[:5].tolist())
).reset_index()
print(f"Venue-level shape: {venue_df.shape}")

venues = venue_df['venue'].tolist()

# average TF-IDF vectors per venue
# .to_numpy() converts pandas boolean Series to numpy array for scipy sparse indexing
venue_tfidf = np.zeros((len(venues), tfidf_matrix.shape[1]))
for i, venue in enumerate(venues):
    mask = (df['venue'] == venue).to_numpy()
    venue_tfidf[i] = np.asarray(tfidf_matrix[mask].mean(axis=0)).flatten()
print(f"Venue TF-IDF matrix shape: {venue_tfidf.shape}")

scaler_z = StandardScaler()
venue_tfidf_zscore = scaler_z.fit_transform(venue_tfidf)
scaler_mm = MinMaxScaler()
venue_tfidf_minmax = scaler_mm.fit_transform(venue_tfidf)

pca = PCA(n_components=2, random_state=42)
venue_pca = pca.fit_transform(venue_tfidf_zscore)
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")

venue_df['pca_x'] = venue_pca[:, 0]
venue_df['pca_y'] = venue_pca[:, 1]

np.save('./data/venue_tfidf.npy', venue_tfidf)
np.save('./data/venue_tfidf_zscore.npy', venue_tfidf_zscore)
np.save('./data/venue_tfidf_minmax.npy', venue_tfidf_minmax)
venue_df.to_parquet('./data/venue_df.parquet')
author_df.to_parquet('./data/author_df.parquet')

print("All outputs saved!")
venue_df.head()

ACL-like venues (before abstract filter):
 venue
meeting of the association for computational linguistics                               4740
international conference on computational linguistics                                  4351
north american chapter of the association for computational linguistics                2679
empirical methods in natural language processing                                       2020
conference of the european chapter of the association for computational linguistics     783
international joint conference on natural language processing                           730
computational linguistics                                                               720
recent advances in natural language processing                                          389
conference on intelligent text processing and computational linguistics                 190
logical aspects of computational linguistics                                            148
Name: count, dtype: int64 

Aft

,venue,paper_count,avg_citations,year_min,year_max,top_authors,pca_x,pca_y
0,computer vision and pattern recognition,3000,101.991333,1988,2017,"[Thomas S. Huang, Takeo Kanade, Xiaoou Tang, T...",-21.055018,-11.803739
1,international conference on machine learning,3000,88.349333,1987,2017,"[Michael I. Jordan, Zoubin Ghahramani, Shie Ma...",-19.994248,-13.060607
2,international conference on management of data,3000,111.758333,1970,2017,"[Michael Stonebraker, Hector Garcia-Molina, Je...",26.327045,0.492855
3,knowledge discovery and data mining,3000,84.565667,1994,2017,"[Jiawei Han, Christos Faloutsos, Philip S. Yu,...",0.785415,-5.049509
4,meeting of the association for computational l...,3000,74.624333,1979,2016,"[Christopher D. Manning, Ido Dagan, Daniel N. ...",-12.625423,33.177842
